In [2]:
import keras.backend as K
import os
import numpy as np
import pylab as plt
from keras.utils import to_categorical
from keras.models import Model
from keras.layers import Input
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers.convolutional import Conv3D
from keras.layers.convolutional_recurrent import ConvLSTM2D
from keras.layers.normalization import BatchNormalization
from keras.models import load_model
from keras.models import load_model
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint
from keras.optimizers import Adam
import matplotlib.pyplot as plt
from keras.utils import to_categorical
from keras.regularizers import l2
from keras.backend import clip 
import math

In [3]:
import os
from keras.utils import multi_gpu_model
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

## Reading Data 

In [4]:
data = np.load('NPY_Files/data_with_frame_splits.npy')
data.shape

(657, 20, 128, 110, 1)

## MinMax Scaling 

In [5]:
from sklearn.preprocessing import minmax_scale

shape = data.shape
data = minmax_scale(data.ravel(), feature_range=(0,255)).reshape(shape)
data.shape

(657, 20, 128, 110, 1)

## Separating the Training and Testing data 

In [6]:
X1_precipitation = data[:100,:16,:,:,:]
X1_precipitation.shape

tem = np.zeros((100,1,128,110,1))
#tem = tem.astype(int)

X2_precipitation = np.concatenate((tem, data[:100,16:19,:,:,:]), axis=1)
#X2_precipitation = X2_precipitation.astype('uint8')
X2_precipitation.shape


y_precipitation = data[:100,16:20,:,:,:]
y_precipitation.shape
    
del tem

## 3 Layer Model Implmentation

In [7]:
"""
3-layer stacked ConvLSTM2D Encoder-Decoder
"""

def define_models_3_precipitation(n_filter, filter_size):
    # define training encoder
    encoder_inputs = Input(shape=(None, 128, 110, 1))
    encoder_1 = ConvLSTM2D(filters = n_filter, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.001), recurrent_regularizer=l2(0.001), bias_regularizer=l2(0.001))
    encoder_2 = ConvLSTM2D(filters = 32, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.001), recurrent_regularizer=l2(0.001), bias_regularizer=l2(0.001))
    encoder_3 = ConvLSTM2D(filters = 32, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.001), recurrent_regularizer=l2(0.001), bias_regularizer=l2(0.001))
    encoder_outputs_1, encoder_state_h_1, encoder_state_c_1 = encoder_1(encoder_inputs)
    encoder_outputs_2, encoder_state_h_2, encoder_state_c_2 = encoder_2(encoder_outputs_1)
    encoder_outputs_3, encoder_state_h_3, encoder_state_c_3 = encoder_3(encoder_outputs_2)
    # define training decoder
    decoder_inputs = Input(shape=(None, 128, 110, 1))
    decoder_1 = ConvLSTM2D(filters=n_filter, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.001), recurrent_regularizer=l2(0.001), bias_regularizer=l2(0.001))
    decoder_2 = ConvLSTM2D(filters=32, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.001), recurrent_regularizer=l2(0.001), bias_regularizer=l2(0.001))
    decoder_3 = ConvLSTM2D(filters=32, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.001), recurrent_regularizer=l2(0.001), bias_regularizer=l2(0.001))
    decoder_outputs_1, _, _ = decoder_1([decoder_inputs, encoder_state_h_1, encoder_state_c_1])
    decoder_outputs_2, _, _ = decoder_2([decoder_outputs_1, encoder_state_h_2, encoder_state_c_2])
    decoder_outputs_3, _, _ = decoder_3([decoder_outputs_2, encoder_state_h_3, encoder_state_c_3])
    decoder_conv3d = Conv3D(filters=1, kernel_size=(1,1,32), activation='relu', padding='same', data_format='channels_last',
                            kernel_regularizer=l2(0.001), bias_regularizer=l2(0.001))
    decoder_outputs = decoder_conv3d(decoder_outputs_3)
    
    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    #print(model.summary(line_length=250))
    # define inference encoder
    encoder_model = Model(encoder_inputs, 
                          [encoder_state_h_1, encoder_state_c_1, encoder_state_h_2, encoder_state_c_2, encoder_state_h_3, encoder_state_c_3])
    
    # define inference decoder
    decoder_state_input_h_1 = Input(shape=(128, 110,n_filter))
    decoder_state_input_c_1 = Input(shape=(128, 110,n_filter))
    decoder_state_input_h_2 = Input(shape=(128, 110,32))
    decoder_state_input_c_2 = Input(shape=(128, 110,32))
    decoder_state_input_h_3 = Input(shape=(128, 110,32))
    decoder_state_input_c_3 = Input(shape=(128, 110,32))
    decoder_output_1, decoder_state_h_1_new, decoder_state_c_1_new = decoder_1([decoder_inputs, decoder_state_input_h_1, decoder_state_input_c_1])
    decoder_output_2, decoder_state_h_2_new, decoder_state_c_2_new = decoder_2([decoder_output_1, decoder_state_input_h_2, decoder_state_input_c_2])
    decoder_output_3, decoder_state_h_3_new, decoder_state_c_3_new = decoder_3([decoder_output_2, decoder_state_input_h_3, decoder_state_input_c_3])
    decoder_output = decoder_conv3d(decoder_output_3)
    decoder_model = Model([decoder_inputs , decoder_state_input_h_1 , decoder_state_input_c_1, decoder_state_input_h_2 , decoder_state_input_c_2, 
                           decoder_state_input_h_3 , decoder_state_input_c_3],
                          [decoder_output, decoder_state_h_1_new, decoder_state_c_1_new, decoder_state_h_2_new, decoder_state_c_2_new, 
                           decoder_state_h_3_new, decoder_state_c_3_new])
    
    return model, encoder_model, decoder_model


In [9]:
train_3_precipitation, infenc_3_precipitation, infdec_3_precipitation = define_models_3_precipitation(n_filter=64, filter_size=3)
train_3_precipitation.compile(loss='mse', optimizer='adam', metrics=['mae'])
filepath = "saved-3LayerconvLSTM-{epoch:02d}.h5"
cp = ModelCheckpoint(filepath, verbose=1, save_best_only=False,mode='max', period=20)
history_3_precipitation = train_3_precipitation.fit([X1_precipitation, X2_precipitation],
                                                    y_precipitation, batch_size=8, validation_split=0.25, epochs=500, callbacks=[cp])


Epoch 1/500
10/10 [==============================] - 13s 1s/step - loss: 16110.2969 - mae: 104.0976 - val_loss: 10231.3760 - val_mae: 94.4278
Epoch 2/500
10/10 [==============================] - 12s 1s/step - loss: 11852.8828 - mae: 104.0776 - val_loss: 9173.2227 - val_mae: 88.5019
Epoch 3/500
10/10 [==============================] - 12s 1s/step - loss: 11831.5654 - mae: 87.8618 - val_loss: 10711.2500 - val_mae: 92.1963
Epoch 4/500
10/10 [==============================] - 12s 1s/step - loss: 9524.3262 - mae: 86.7296 - val_loss: 8227.7910 - val_mae: 78.0341
Epoch 5/500
10/10 [==============================] - 12s 1s/step - loss: 6819.5205 - mae: 66.0921 - val_loss: 5938.8765 - val_mae: 58.6927
Epoch 6/500
10/10 [==============================] - 12s 1s/step - loss: 6403.2363 - mae: 62.2014 - val_loss: 6136.9404 - val_mae: 57.8133
Epoch 7/500
10/10 [==============================] - 12s 1s/step - loss: 10148.9287 - mae: 76.2668 - val_loss: 11569.7227 - val_mae: 103.0608
Epoch 8/500
10/10

KeyboardInterrupt: 